# 04 — Topic Modelling with BERTopic

Inputs: `../data/grab_reviews_with_predictions.csv`  
Outputs: `../data/grab_reviews_topics.csv`

Steps:
1. Fit BERTopic on cleaned reviews
2. Inspect and label discovered topics (GrabFood, GrabCar, GrabPay, etc.)
3. Cross-tabulate sentiment × topic
4. Plot sentiment trend over time per topic

In [1]:
import warnings, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/grab_reviews_with_predictions.csv', parse_dates=['date'])
print(f'Loaded {len(df):,} reviews')
df.head(2)

[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.3.1+cpu
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


NameError: name 'nn' is not defined

## 1. Fit BERTopic

In [ ]:
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

docs = df['review_clean'].tolist()

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

umap_model = UMAP(
    n_neighbors=15, n_components=5,
    min_dist=0.0, metric='cosine', random_state=42
)
hdbscan_model = HDBSCAN(
    min_cluster_size=50, metric='euclidean',
    cluster_selection_method='eom', prediction_data=True
)
vectorizer = CountVectorizer(
    stop_words='english', ngram_range=(1, 2), min_df=5
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    top_n_words=10,
    verbose=True,
)

topics, probs = topic_model.fit_transform(docs)
df['topic_id'] = topics

print(f'Discovered {topic_model.get_topic_info().shape[0] - 1} topics (excl. noise)')

## 2. Inspect Topics

In [ ]:
topic_info = topic_model.get_topic_info()
topic_info.head(20)

In [ ]:
# Print top words per topic to help assign business labels
for tid in topic_info['Topic'].head(15):
    if tid == -1:
        continue
    words = [w for w, _ in topic_model.get_topic(tid)]
    print(f'Topic {tid:3d}: {words}')

In [ ]:
# Manually assign business-friendly labels after inspecting the words above
# Adjust this mapping based on your actual discovered topics
TOPIC_LABELS = {
    -1: 'Noise / Uncategorised',
    0:  'GrabFood — Delivery',
    1:  'GrabCar — Driver Quality',
    2:  'App Performance / Bugs',
    3:  'GrabPay — Payment Issues',
    4:  'Pricing / Surge',
    5:  'Customer Service',
    6:  'GrabBike',
    7:  'Promotions / Rewards',
    8:  'Waiting Time / ETA',
    9:  'Safety',
}

df['topic_label'] = df['topic_id'].map(TOPIC_LABELS).fillna(df['topic_id'].apply(lambda x: f'Topic {x}'))

print(df['topic_label'].value_counts().head(15))

## 3. Sentiment × Topic Cross-Tabulation

In [ ]:
pivot = (
    df[df['topic_id'] != -1]
    .groupby(['topic_label', 'bert_pred'])
    .size()
    .unstack(fill_value=0)
    .assign(total=lambda x: x.sum(axis=1))
    .sort_values('total', ascending=False)
)

# Percentage share
for col in ['negative', 'neutral', 'positive']:
    if col in pivot.columns:
        pivot[f'{col}_pct'] = (pivot[col] / pivot['total'] * 100).round(1)

print(pivot[['negative', 'neutral', 'positive', 'total']].head(15))

In [ ]:
top_topics = pivot.head(10).index.tolist()
plot_df = (
    df[(df['topic_id'] != -1) & (df['topic_label'].isin(top_topics))]
    .groupby(['topic_label', 'bert_pred'])
    .size()
    .reset_index(name='count')
)

palette = {'negative': '#e74c3c', 'neutral': '#f39c12', 'positive': '#2ecc71'}

fig, ax = plt.subplots(figsize=(14, 6))
pivot_plot = plot_df.pivot(index='topic_label', columns='bert_pred', values='count').fillna(0)
pivot_plot = pivot_plot.reindex(top_topics)

cols_order = [c for c in ['negative', 'neutral', 'positive'] if c in pivot_plot.columns]
pivot_plot[cols_order].plot(
    kind='bar', stacked=True, ax=ax,
    color=[palette[c] for c in cols_order], edgecolor='white'
)
ax.set_title('Review Sentiment by Topic (Top 10)', fontsize=14)
ax.set_xlabel('Topic')
ax.set_ylabel('Review Count')
ax.legend(title='Sentiment')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../data/04_sentiment_by_topic.png', dpi=120)
plt.show()

## 4. Sentiment Trend Over Time per Topic

In [ ]:
sentiment_score = {'negative': -1, 'neutral': 0, 'positive': 1}
df['sent_score'] = df['bert_pred'].map(sentiment_score)

# Select top 5 topics (by volume, excluding noise)
top5 = (
    df[df['topic_id'] != -1]['topic_label']
    .value_counts()
    .head(5)
    .index.tolist()
)

fig, ax = plt.subplots(figsize=(14, 6))

for topic in top5:
    ts = (
        df[df['topic_label'] == topic]
        .set_index('date')
        .resample('ME')['sent_score']
        .mean()
        .rolling(3, min_periods=1)
        .mean()
    )
    ax.plot(ts.index, ts.values, marker='o', ms=3, lw=1.5, label=topic)

ax.axhline(0, color='black', lw=0.8, linestyle='--', alpha=0.4)
ax.set_title('Sentiment Trend Over Time by Topic (3-month rolling avg)', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Avg Sentiment Score (-1 neg → +1 pos)')
ax.legend(loc='lower left', fontsize=9)
ax.set_ylim(-1, 1)
plt.tight_layout()
plt.savefig('../data/04_sentiment_trend_by_topic.png', dpi=120)
plt.show()

## 5. Save Final Dataset

In [ ]:
OUT = '../data/grab_reviews_topics.csv'
df.to_csv(OUT, index=False)
print(f'Saved {len(df):,} rows to {OUT}')
df[['review_clean', 'sentiment', 'bert_pred', 'topic_label', 'date']].head()